[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GPXCAT/docker_whisper/blob/main/whisper.ipynb)

顯示顯示卡資訊

In [ ]:
!nvidia-smi

環境安裝

In [ ]:
!pip install paddlepaddle-gpu paddleocr>=2.0.1 opencv-python srt

In [ ]:
!pip list | grep paddle

In [ ]:
import paddle
print(paddle.device.get_device())  # 如果有成功啟動GPU應該會返回 'gpu:0'

In [ ]:
!apt update && apt install -y ffmpeg ccache

掛載GoogleDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

lang must in dict_keys(['ch', 'ch_doc', 'en', 'korean', 'japan', 'chinese_cht', 'ta', 'te', 'ka', 'latin', 'arabic', 'cyrillic', 'devanagari'])

In [ ]:
import argparse
import copy
import datetime

import cv2
from paddleocr import PaddleOCR
from skimage.metrics import structural_similarity
import srt

import sys
filename = 'ave_mujica_03'
# 模擬命令列參數（第一個是 script 名稱，之後是參數）
sys.argv = ['colab_kernel_launcher.py', '--lang=chinese_cht',
            f'/content/drive/MyDrive/{filename}.mp4',
            f'/content/drive/MyDrive/{filename}.srt',
            f'/content/drive/MyDrive/{filename}.txt']

def box2int(box):
    for i in range(len(box)):
        for j in range(len(box[i])):
            box[i][j] = int(box[i][j])
    return box


def detect_subtitle_area(ocr_results, h, w):
    '''
    Args:
        w(int): width of the input video
        h(int): height of the input video
    '''
    ocr_results = ocr_results[0]  # 0, the first image result
    # Merge horizon text areas
    idx = 0
    candidates = []
    if ocr_results is not None:
        while idx < len(ocr_results):
            boxes, text = ocr_results[idx]
            # We assume the subtitle is at bottom of the video
            if boxes[0][1] < h * 0.75:
                idx += 1
                continue
            idx += 1
            con_boxes = copy.deepcopy(boxes)
            con_text = text[0]
            while idx < len(ocr_results):
                n_boxes, n_text = ocr_results[idx]
                if abs(n_boxes[0][1] - boxes[0][1]) < h * 0.01 and \
                abs(n_boxes[3][1] - boxes[3][1]) < h * 0.01:
                    con_boxes[1] = n_boxes[1]
                    con_boxes[2] = n_boxes[2]
                    con_text = con_text + ' ' + n_text[0]
                    idx += 1
                else:
                    break
            candidates.append((con_boxes, con_text))
    # TODO(Binbin Zhang): Only support horion center subtitle
    if len(candidates) > 0:
        sub_boxes, subtitle = candidates[-1]
        # offset is less than 10%
        if (sub_boxes[0][0] + sub_boxes[1][0]) / w > 0.90:
            return True, box2int(sub_boxes), subtitle
    return False, None, None


def get_args():
    parser = argparse.ArgumentParser(description='we subtitle')
    parser.add_argument('-s',
                        '--subsampling',
                        type=int,
                        default=3,
                        help='subsampling rate, for speedup')
    parser.add_argument('-t',
                        '--similarity_thresh',
                        type=float,
                        default=0.8,
                        help='similarity threshold')
    parser.add_argument('-l',
                        '--lang',
                        type=str,
                        default='chinese_cht',
                        help='language default=chinese_cht')
    parser.add_argument('input_video', help='input video file')
    parser.add_argument('output_srt', help='output srt file')
    parser.add_argument('output_txt', help='output txt file')
    args = parser.parse_args()
    return args

def extract_gray_region(frame, box):
    """
    根據 box 座標安全擷取灰階區塊
    """
    x_coords = [point[0] for point in box]
    y_coords = [point[1] for point in box]
    x1, x2 = int(min(x_coords)), int(max(x_coords))
    y1, y2 = int(min(y_coords)), int(max(y_coords))

    # 安全檢查座標
    if x2 <= x1 or y2 <= y1:
        print(f"警告：無效的 box 區域，跳過：{box}")
        return None

    region = frame[y1:y2, x1:x2]
    if region.size == 0:
        print(f"警告：擷取到空區域，跳過：{box}")
        return None

    return cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)

def main():
    args = get_args()
    ocr = PaddleOCR(use_angle_cls=True, lang=args.lang, use_gpu=True)
    cap = cv2.VideoCapture(args.input_video)
    w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    fps = cap.get(cv2.CAP_PROP_FPS)
    print('Video info w: {}, h: {}, count: {}, fps: {}'.format(
        w, h, count, fps))

    cur = 0
    detected = False
    box = None
    content = ''
    start = 0
    ref_gray_image = None
    subs = []

    def _add_subs(end):
        print('New subtitle {} {} {}'.format(start / fps, end / fps, content))
        subs.append(
            srt.Subtitle(
                index=0,
                start=datetime.timedelta(seconds=start / fps),
                end=datetime.timedelta(seconds=end / fps),
                content=content.strip(),
            ))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            if detected:
                _add_subs(cur)
            break
        cur += 1
        if cur % args.subsampling != 0:
            continue
        if detected:
            # Compute similarity to reference subtitle area, if the result is
            # bigger than thresh, it's the same subtitle, otherwise, there is
            # changes in subtitle area
            hyp_gray_image = extract_gray_region(frame, box)
            if hyp_gray_image is None:
                continue
            similarity = structural_similarity(hyp_gray_image, ref_gray_image)
            if similarity > args.similarity_thresh:  # the same subtitle
                continue
            else:
                # Record current subtitle
                _add_subs(cur - args.subsampling)
                detected = False
        else:
            # Detect subtitle area
            ocr_results = ocr.ocr(frame)
            detected, box, content = detect_subtitle_area(ocr_results, h, w)
            if detected:
                start = cur
                ref_gray_image = extract_gray_region(frame, box)
                if ref_gray_image is None:
                    continue
    cap.release()

    # Write srt file
    with open(args.output_srt, 'w', encoding='utf8') as fout:
        fout.write(srt.compose(subs))

    # Write txt file
    with open(args.output_txt, 'w', encoding='utf8') as fout:
        for idx in range(len(subs)):
            # same subtitle occurs once
            if idx > 0:
                if subs[idx].content == subs[idx-1].content:
                    continue

            fout.write(subs[idx].content + "\n")

if __name__ == '__main__':
    main()
